# Сравнение двух новых политик на одних логах

Пример использует симулятор OffPolicyLab и зафиксированную версию библиотеки. Для запуска установите `policyscope @ git+https://github.com/Yurashku/OffPolicyLab.git@7ded1b1691c94babf531a23198720605cc29d781`. Логи создаются локально. Истинные эффекты известны только в симуляции.

In [1]:
import policyscope as ps

sim = ps.synthetic.OfferSimulator(seed=0)
clients = sim.sample_clients(4000)
incumbent = sim.policy("epsilon_greedy", epsilon=0.2, seed=1)
logs = sim.simulate(clients, incumbent)
features = ["age", "income", "loyal", "risk", "region"]
print(f"Решений в логах: {len(logs)}")

Решений в логах: 4000


In [2]:
for candidate_seed, temperature in [(2, 0.6), (3, 1.5)]:
    candidate = sim.policy("softmax", temperature=temperature, seed=candidate_seed)
    report = ps.evaluate(logs, candidate, action="offer", reward=["accepted", "revenue"],
                         propensity="propensity", features=features, random_state=0, n_jobs=1)
    print(f"Кандидат {candidate_seed}, сравнение с действующей политикой")
    for metric in ("accepted", "revenue"):
        result = report[metric]
        truth = sim.true_value(clients, candidate, metric) - sim.true_value(clients, incumbent, metric)
        print(f"  {metric}: оценка {result.delta.value:.4f}, "
              f"95% ДИ [{result.delta.low:.4f}, {result.delta.high:.4f}], "
              f"истина {truth:.4f}, диагностика {result.trust}")

Кандидат 2, сравнение с действующей политикой
  accepted: оценка 0.0369, 95% ДИ [0.0010, 0.0729], истина 0.0475, диагностика reliable
  revenue: оценка 10.0652, 95% ДИ [2.3295, 17.8008], истина 14.9936, диагностика reliable
Кандидат 3, сравнение с действующей политикой
  accepted: оценка -0.0811, 95% ДИ [-0.1135, -0.0487], истина -0.0558, диагностика reliable
  revenue: оценка -4.1038, 95% ДИ [-11.0290, 2.8214], истина 2.3434, диагностика reliable


## Интерпретация

Доверительные интервалы выше относятся к отдельным сравнениям с действующей политикой. Для вывода о превосходстве одной новой версии над другой требуется соответствующий контраст с учётом совместной неопределённости. Доли ненадёжных прогонов на рисунке принятого отчёта получены в отдельной [серии симуляций](https://github.com/Yurashku/OffPolicyLab/blob/2d553ac5362263c9bad6a3f827e4a21dc3e79ab0/docs/validation.md). Ускорение выпуска и экономию на A/B-тестах предстоит измерить при внедрении.